<a href="https://colab.research.google.com/github/Bosmithan/Code/blob/main/Authoraldentification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [ ]:
import re
import numpy as np

import nltk
from nltk.corpus import reuters
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

Load the Reuters data set

In [ ]:
nltk.download('reuters')

documents = reuters.fileids()[:300]

[nltk_data] Downloading package reuters to /root/nltk_data...


Add (generate) the author lable as the data set unlable

In [ ]:
texts = []
authors = []

for i, doc_id in enumerate(documents):
    text = reuters.raw(doc_id)
    print(text)
    texts.append(text)

    if i < 100:
        authors.append("Author_A")
    elif i < 200:
        authors.append("Author_B")
    else:
        authors.append("Author_C")

ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT
  Mounting trade friction between the
  U.S. And Japan has raised fears among many of Asia's exporting
  nations that the row could inflict far-reaching economic
  damage, businessmen and officials said.
      They told Reuter correspondents in Asian capitals a U.S.
  Move against Japan might boost protectionist sentiment in the
  U.S. And lead to curbs on American imports of their products.
      But some exporters said that while the conflict would hurt
  them in the long-run, in the short-term Tokyo's loss might be
  their gain.
      The U.S. Has said it will impose 300 mln dlrs of tariffs on
  imports of Japanese electronics goods on April 17, in
  retaliation for Japan's alleged failure to stick to a pact not
  to sell semiconductors on world markets at below cost.
      Unofficial Japanese estimates put the impact of the tariffs
  at 10 billion dlrs and spokesmen for major electronics firms
  said they would virtually halt exports

Process the  tex

1.   Convert the lower case
2.   Remove special characters
3.   lemmatize the token using WordNet

In [ ]:
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

texts_clean = [preprocess(text) for text in texts]

[nltk_data] Downloading package wordnet to /root/nltk_data...


Feature Extraction using TF-ID on both unigrams and bigrams

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words='english',
    max_features=5000
)

X_tfidf = vectorizer.fit_transform(texts_clean)



Add stylometric features

1.  Sentence length
2.  Use of digits
3.  Use exclamation



In [ ]:
def extract_style_features(text):
    num_commas = len(re.findall(r',', text))
    num_exclamations = len(re.findall(r'!', text))
    num_digits = len(re.findall(r'\d', text))

    words = text.split()
    avg_word_length = np.mean([len(w) for w in words]) if words else 0
    uppercase_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)

    return [
        num_commas,
        num_exclamations,
        num_digits,
        avg_word_length,
        uppercase_ratio
    ]

style_features = np.array(
    [extract_style_features(text) for text in texts]
)

Concat TF-IDF and stylometric features

In [ ]:
X = hstack([X_tfidf, style_features])
y = authors

Build the classification model
Train:Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,
    random_state=42
)


Build the classification model: Creat a logistic regration model and train it.

In [ ]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Predict(test) the model

In [ ]:
predictions = classifier.predict(X_test)

Evaluvate the mode by computing the perrformance matrix(accuracy, F1)

In [ ]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

    Author_A       0.53      0.36      0.43        22
    Author_B       0.40      0.62      0.49        16
    Author_C       0.70      0.64      0.67        22

    accuracy                           0.53        60
   macro avg       0.54      0.54      0.53        60
weighted avg       0.56      0.53      0.53        60

